# Teks — Fine-tune transformer k-fold + OOF (jalur skor tertinggi)
**Kapan dipakai:** klasifikasi teks (satu kolom, atau pasangan `TEXT_A` + `TEXT_B`), ada GPU, dan
sudah ada baseline dari `baseline_oof`. Seluruh bobot model ikut dilatih ulang, sekali per fold.
Probabilitas OOF dan test disimpan ke `OOF_FOLDER` untuk `ensemble_oof`.
`klasifikasi_finetune` tetap jalur cepat (satu split, tanpa OOF).

**Alur:** Konfigurasi → baca → bersihkan → panjang token → k-fold → latih tiap fold (AMP bf16,
padding dinamis, warmup linear, simpan bobot terbaik) → OOF → simpan → submission → validator.

**Saat lomba yang diubah:** sel Konfigurasi (`# GANTI saat lomba`). Waktu mepet:
`FOLDS_TO_RUN = [0]` (satu fold, OOF tidak disimpan). Alternatif ber-`#` ada di sel Konfigurasi
dan Opsi: hapus `#` pada baris yang dipakai, beri `#` pada baris aslinya. Hapus sel bertanda
`HAPUS saat lomba`.

**Data demo:** SmSA (`sentiment`, 3 kelas). Bentuk lain hanya dengan mengubah Konfigurasi:
`nusax` dengan `xlmr_base`, `pair` (wrete) dengan `TEXT_A="sent_A"` dan `TEXT_B="sent_B"`.

In [ ]:
import html
import os
import re
import time

import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, StratifiedKFold
from torch.nn.functional import cross_entropy, softmax
from torch.nn.utils import clip_grad_norm_
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import get_linear_schedule_with_warmup

## Konfigurasi
Data dan kolom (sel ini), model dan hiperparameter (sel berikut), alternatif lanjutan (sel Opsi).
`OOF_FOLDER` dipakai bersama notebook lain: `oof_<MODEL_NAME>.npy` dan `test_<MODEL_NAME>.npy`.

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/finetune_oof"
OOF_FOLDER = "../../../outputs/text/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
SUBMISSION_NAME = "submission.csv"                             # GANTI saat lomba bila diminta lain
ANSWER_CSV = f"{DATA_FOLDER}/smsa_test_labels.csv"             # HAPUS saat lomba (kunci demo)

ID_COL = "id"                    # GANTI saat lomba
TEXT_A = "text"                  # GANTI saat lomba: kolom teks (pasangan: kalimat pertama)
TEXT_B = None                    # GANTI: kolom kalimat kedua untuk pasangan, None = teks tunggal
# TEXT_B = "sent_B"              # pasangan teks (NLI, paraphrase, relevansi): TEXT_A = "sent_A"
VERBAL_COLS = []                 # teks+tabel: kolom lain ditulis di depan teks "kolom: nilai | "
# VERBAL_COLS = ["title", "division_name", "age"]   # BERT ikut membaca kolom tabel (demo baju)
TARGET = "label"                 # GANTI saat lomba
SUBMIT_COL = TARGET              # GANTI saat lomba bila kolom prediksi di sample bernama lain
GROUP_COL = None                 # GANTI: kolom penulis/entitas untuk GroupKFold (None = tidak)
N_FOLDS = 5
SPLIT_SEED = 42                  # seed split: tetap 42 agar OOF sefold dengan model lain

In [ ]:
PRETRAINED = "../../../pretrained/indobert"                  # GANTI saat lomba
# PRETRAINED = "../../../pretrained/indobert_large"   # Indonesia formal, terkuat: GRAD_CHECKPOINT
# PRETRAINED = "../../../pretrained/indobertweet"     # medsos Indonesia: WAJIB prep_tweet
# PRETRAINED = "../../../pretrained/xlmr_base"        # bahasa daerah / campur / Inggris
# PRETRAINED = "../../../pretrained/nusabert"         # Indonesia + bahasa daerah (NusaX)
# PRETRAINED = "../../../pretrained/nli_mdeberta"     # backbone NLI multibahasa: pasangan
# PRETRAINED = "../../../pretrained/distilbert_id"    # Indonesia, 2x lebih cepat
# PRETRAINED = "../../../pretrained/minilm"           # multibahasa kecil, tercepat (waktu mepet)
MODEL_NAME = "ft_indobert"       # GANTI tiap variasi (nama file OOF, jangan menimpa model lain)
MAX_LENGTH = 128                 # GANTI: lihat persentil token di bawah (p95 cukup)
BATCH_SIZE = 16                  # 8 untuk large. Batch efektif = BATCH_SIZE x ACCUM_STEPS
ACCUM_STEPS = 1                  # 2 bila BATCH_SIZE diturunkan agar batch efektif tetap 16
EPOCHS = 3
LEARNING_RATE = 3e-5             # 1e-5 .. 5e-5 (large: 1e-5 .. 2e-5)
WARMUP = 0.1                     # bagian awal langkah dengan lr naik, lalu turun linear
WEIGHT_DECAY = 0.01
FOLDS_TO_RUN = None              # None = semua fold. Waktu mepet: [0] (OOF tidak disimpan)
# FOLDS_TO_RUN = [0]
SEEDS = [42]                     # seed inisialisasi kepala. Rata-rata antar seed: [42, 7]
# SEEDS = [42, 7]                # data kecil (< 1000): hasil beda jauh antar seed
EVALS_PER_EPOCH = 1              # validasi tiap epoch
# EVALS_PER_EPOCH = 2            # data besar / waktu mepet: lebih sering, cocok dengan PATIENCE
PATIENCE = 0                     # 0 = latih semua epoch
# PATIENCE = 2                   # berhenti bila 2 validasi berturut-turut tidak naik
SELECT_BEST = True               # bobot terbaik di validasi fold (sedikit optimistis untuk OOF)
# SELECT_BEST = False            # bobot terakhir saja: OOF tanpa bias seleksi
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"      # bf16: RTX 30xx ke atas. fp16 tidak dipakai (butuh GradScaler)

## Opsi lanjutan (semua mati; hapus `#` pada yang dipakai)
Satu per satu atau digabung. Biaya waktu ada di komentar. Semua diuji jalan, tak semuanya menang.

In [ ]:
HEAD_TAIL = 0                    # 0 = potong di akhir teks (teks tunggal)
# HEAD_TAIL = 32                 # teks panjang, info di akhir (ulasan): awal + 32 token akhir
GRAD_CHECKPOINT = False
# GRAD_CHECKPOINT = True         # model large di 8 GB: hemat VRAM, ± 30% lebih lambat
LOSS_NAME = "ce"                 # cross-entropy
# LOSS_NAME = "focal"            # kelas langka / contoh sulit (FOCAL_GAMMA)
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.0
# LABEL_SMOOTHING = 0.1          # label berisik
CLASS_WEIGHT = False
# CLASS_WEIGHT = True            # kelas timpang: bobot = n / (K x jumlah kelas) di data latih
LAYER_DECAY = 1.0                # 1.0 = semua lapisan lr sama
# LAYER_DECAY = 0.9              # LLRD: lapisan bawah lr lebih kecil, fine-tune lebih stabil
REINIT_LAYERS = 0
# REINIT_LAYERS = 2              # reset 2 lapisan atas ke acak: data kecil, model besar
FGM_EPS = 0.0
# FGM_EPS = 0.5                  # FGM: latih ulang dengan embedding diganggu, ± 50% lebih lambat
EMA_DECAY = 0.0
# EMA_DECAY = 0.99               # EMA bobot (0.99 untuk < 2000 langkah, 0.999 bila lebih)
PSEUDO_CSV = None
# PSEUDO_CSV = "../../../outputs/text/ensemble_oof/pseudo_label.csv"  # kolom ID_COL, TARGET

## Metrik
Dipakai untuk memilih bobot terbaik per fold, skor OOF, dan cek akhir. Semua di dalam memakai
indeks kelas 0..K-1 (urut `np.unique` label train). Ganti baris `return` bila metrik panitia lain.

In [ ]:
METRIC_NAME = "f1_macro"         # GANTI saat lomba, ikut baris return di bawah


def score(truth, predicted):
    return f1_score(truth, predicted, average="macro")
    # return f1_score(truth, predicted, average="weighted")              # F1 berbobot
    # return f1_score(truth, predicted, pos_label=1)     # biner: F1 kelas ke-2 (indeks 1)


print("metrik:", METRIC_NAME, "| device:", DEVICE)

## Baca data
Teks kosong/NaN jadi string kosong. `train_a`/`test_a` = kolom teks pertama, `train_b`/`test_b` =
kedua (None bila teks tunggal).

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
train_a = train[TEXT_A].fillna("").astype(str)
test_a = test[TEXT_A].fillna("").astype(str)
for col in reversed(VERBAL_COLS):
    train_a = col + ": " + train[col].fillna("kosong").astype(str) + " | " + train_a
    test_a = col + ": " + test[col].fillna("kosong").astype(str) + " | " + test_a
train_b = None
test_b = None
if TEXT_B is not None:
    train_b = train[TEXT_B].fillna("").astype(str)
    test_b = test[TEXT_B].fillna("").astype(str)
print("train:", train.shape, "| test:", test.shape, "| NaN target:", train[TARGET].isna().sum())
train[[ID_COL, TEXT_A, TARGET]].head()

## Buat sample_submission demo (HAPUS saat lomba)
Data latihan tidak punya `sample_submission`. Saat lomba file panitia dipakai (`SAMPLE_CSV`);
sel ini tidak menimpa file di luar `OUTPUT_FOLDER`.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUBMIT_COL] = train[TARGET].iloc[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
else:
    print("SAMPLE_CSV bukan file demo: dibiarkan apa adanya")

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist())
print("id test = id sample:", set(test[ID_COL]) == set(sample[ID_COL]))
sample.head(3)

## Bersihkan teks
Aktif: `html.unescape` dan spasi rapi saja (huruf besar/kecil diurus tokenizer). Model transformer
tidak perlu stopword/stemming. `prep_tweet` WAJIB bila `indobertweet` (sesuai model card): huruf
kecil, mention jadi `@USER`, URL jadi `HTTPURL`, emoji jadi teks.

In [ ]:
def clean_basic(text):
    return " ".join(html.unescape(text).split())


def prep_tweet(text):
    text = clean_basic(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", "HTTPURL", text)
    text = re.sub(r"@\w+", "@USER", text)
    return emoji.demojize(text)


print(prep_tweet("Mantap!! cek https://contoh.com @admin 😀"))
train_a = train_a.apply(clean_basic)
test_a = test_a.apply(clean_basic)
# train_a = train_a.apply(prep_tweet)    # indobertweet (dan teks medsos pada model lain)
# test_a = test_a.apply(prep_tweet)

## Tokenizer dan panjang token
Pilih `MAX_LENGTH` sekitar persentil 95-99 (teks lebih panjang dipotong). Peringatan dicetak bila
`MAX_LENGTH` di bawah p95. Padding dinamis: tiap batch hanya selebar teks terpanjangnya.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED, local_files_only=True)
# token khusus depan/belakang ([CLS]..[SEP] atau <s>..</s>), dibaca dari contoh singkat
plain_ids = tokenizer("a", add_special_tokens=False)["input_ids"]
full_ids = tokenizer("a")["input_ids"]
cut = full_ids.index(plain_ids[0])
SPECIAL_PREFIX = full_ids[:cut]
SPECIAL_SUFFIX = full_ids[cut + len(plain_ids):]


def head_tail_rows(texts):
    token_ids = tokenizer(list(texts), add_special_tokens=False)["input_ids"]
    limit = MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False)
    rows = []
    for ids in token_ids:
        if len(ids) > limit:
            ids = ids[:limit - HEAD_TAIL] + ids[-HEAD_TAIL:]
        full = SPECIAL_PREFIX + ids + SPECIAL_SUFFIX
        rows.append({"input_ids": full, "attention_mask": [1] * len(full)})
    return rows


print("token khusus:", SPECIAL_PREFIX, SPECIAL_SUFFIX)

In [ ]:
def encode_rows(texts_a, texts_b, truncate=True):
    if HEAD_TAIL > 0 and texts_b is None and truncate:
        return head_tail_rows(texts_a)
    second = None
    if texts_b is not None:
        second = list(texts_b)
    encoded = tokenizer(list(texts_a), second, truncation=truncate, max_length=MAX_LENGTH)
    rows = []
    for number in range(len(encoded["input_ids"])):
        row = {}
        for key in encoded.keys():
            row[key] = encoded[key][number]
        rows.append(row)
    return rows


print("encode_rows siap")

In [ ]:
head_b = None
if train_b is not None:
    head_b = train_b.head(2000)
lengths = []
for row in encode_rows(train_a.head(2000), head_b, truncate=False):
    lengths.append(len(row["input_ids"]))
p50, p95, p99 = np.percentile(lengths, [50, 95, 99])
print(f"token per teks: p50 {p50:.0f} | p95 {p95:.0f} | p99 {p99:.0f} | maks {max(lengths)}")
print(f"MAX_LENGTH {MAX_LENGTH}: teks terpotong {(np.array(lengths) > MAX_LENGTH).mean():.1%}")
if MAX_LENGTH < p95:
    print("PERHATIAN: MAX_LENGTH di bawah p95, naikkan (VRAM dan waktu naik)")
print("teks kosong: train", (train_a == "").sum(), "| test", (test_a == "").sum())
print(train[TARGET].value_counts().to_dict())

## Split (k-fold)
Aturan format OOF: `StratifiedKFold` acak seed `SPLIT_SEED`, urutan baris train apa adanya. Ganti
`splitter` hanya bila ada temuan di `baseline_oof` (penulis sama di train dan test → Group); OOF
lalu tidak sefold dengan model lain.

In [ ]:
classes = np.unique(train[TARGET])
y = np.searchsorted(classes, train[TARGET].to_numpy())
groups = train.get(GROUP_COL)      # None bila GROUP_COL = None
# groups = pd.factorize(train_a)[0]   # teks kembar satu lipatan: + StratifiedGroupKFold
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
# splitter = GroupKFold(n_splits=N_FOLDS)     # penulis/entitas sama di train & test: isi GROUP_COL
# splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
folds = list(splitter.split(train_a, y, groups))
split_name = type(splitter).__name__
val_sizes = []
for _, val_index in folds:
    val_sizes.append(len(val_index))
print("kelas:", list(classes), "| fold:", len(folds), "| ukuran validasi:", val_sizes)

## Pseudo-label (opsional, `PSEUDO_CSV`)
CSV berkolom `ID_COL` dan `TARGET` (baris test yang yakin, ditulis `ensemble_oof`). Barisnya
ditambahkan ke data latih tiap fold, tidak pernah ke validasi, jadi OOF tetap bersih.

In [ ]:
train_rows = encode_rows(train_a, train_b)
test_rows = encode_rows(test_a, test_b)
extra_rows = []
extra_y = np.array([], dtype=int)
if PSEUDO_CSV is not None:
    pseudo = pd.read_csv(PSEUDO_CSV)
    chosen = np.where(test[ID_COL].isin(pseudo[ID_COL]))[0]
    label_by_id = pd.Series(pseudo[TARGET].to_numpy(), index=pseudo[ID_COL])
    extra_y = np.searchsorted(classes, label_by_id.reindex(test[ID_COL].iloc[chosen]).to_numpy())
    for number in chosen:
        extra_rows.append(test_rows[number])
print("baris train:", len(train_rows), "| test:", len(test_rows), "| pseudo:", len(extra_rows))

## Model
`make_model`: transformer pretrained + kepala klasifikasi baru (peringatan "newly initialized"
itu normal). `param_groups`: lr per lapisan (`LAYER_DECAY`). Semua dari `pretrained/` lewat path
lokal.

In [ ]:
def reinit_top_layers(model):
    if REINIT_LAYERS > 0:
        for layer in model.base_model.encoder.layer[-REINIT_LAYERS:]:
            for module in layer.modules():
                model._init_weights(module)


def make_model(seed):
    torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED, num_labels=len(classes), ignore_mismatched_sizes=True, local_files_only=True,
        dtype=torch.float32)       # paksa fp32: nli_mdeberta tersimpan fp16, latih fp16 murni = NaN
    if GRAD_CHECKPOINT:
        model.gradient_checkpointing_enable()
    reinit_top_layers(model)
    return model.to(DEVICE)


print("make_model siap:", PRETRAINED)

In [ ]:
def add_group(groups, params, scale, used):
    fresh = []
    for param in params:
        if id(param) not in used:
            used.add(id(param))
            fresh.append(param)
    if fresh:
        groups.append({"params": fresh, "lr": LEARNING_RATE * scale})


def param_groups(model):
    layers = model.base_model.encoder.layer
    groups, used = [], set()
    for number, layer in enumerate(layers):
        add_group(groups, layer.parameters(), LAYER_DECAY ** (len(layers) - number), used)
    embed_scale = LAYER_DECAY ** (len(layers) + 1)
    add_group(groups, model.base_model.embeddings.parameters(), embed_scale, used)
    add_group(groups, model.parameters(), 1.0, used)     # kepala dan sisanya: lr penuh
    return groups


print("param_groups siap, LAYER_DECAY", LAYER_DECAY)

In [ ]:
def compute_loss(logits, targets, weight):
    loss = cross_entropy(logits, targets, weight=weight, label_smoothing=LABEL_SMOOTHING,
                         reduction="none")
    if LOSS_NAME == "focal":
        plain = cross_entropy(logits, targets, reduction="none")
        loss = loss * (1 - torch.exp(-plain)) ** FOCAL_GAMMA
    return loss.mean()


def fgm_attack(model):
    weight = model.get_input_embeddings().weight
    backup = weight.data.clone()
    norm = torch.norm(weight.grad)
    if norm > 0 and torch.isfinite(norm):
        weight.data.add_(FGM_EPS * weight.grad / norm)
    return backup


print("loss:", LOSS_NAME, "| FGM_EPS:", FGM_EPS)

In [ ]:
def ema_update(shadow, model):
    with torch.no_grad():
        for name, param in model.named_parameters():
            shadow[name].mul_(EMA_DECAY).add_(param.detach(), alpha=1 - EMA_DECAY)


def ema_swap(shadow, model):
    with torch.no_grad():
        for name, param in model.named_parameters():
            kept = param.detach().clone()
            param.copy_(shadow[name])
            shadow[name].copy_(kept)


def snapshot(model):
    state = {}
    for key, tensor in model.state_dict().items():
        moved = tensor.detach().cpu()
        state[key] = moved.clone()
    return state


print("EMA_DECAY:", EMA_DECAY)

In [ ]:
def pick(rows, chosen):
    picked = []
    for number in chosen:
        picked.append(rows[number])
    return picked


def forward_logits(model, rows):
    batch = tokenizer.pad(rows, return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
        return model(**batch).logits.float()


def predict_proba(model, rows):
    model.eval()
    sizes = []
    for row in rows:
        sizes.append(len(row["input_ids"]))
    order = np.argsort(sizes)           # urut panjang: padding minimal
    proba = np.zeros((len(rows), len(classes)), dtype="float32")
    with torch.no_grad():
        for start in range(0, len(order), BATCH_SIZE * 4):
            chosen = order[start:start + BATCH_SIZE * 4]
            logits = forward_logits(model, pick(rows, chosen))
            proba[chosen] = softmax(logits, dim=1).cpu().numpy()
    return proba


print("pick/forward/predict siap")

## Latih satu fold
`run` = kantong keadaan satu latihan (model, optimizer, jadwal, EMA, bobot terbaik). Setiap
`micro_step` satu batch; tiap `ACCUM_STEPS` batch satu langkah optimizer (clip grad 1.0). Validasi
dihitung dengan bobot EMA bila `EMA_DECAY > 0`.

In [ ]:
def micro_step(run, rows, targets):
    model = run["model"]
    target_tensor = torch.tensor(targets).to(DEVICE)
    loss = compute_loss(forward_logits(model, rows), target_tensor, run["weight"])
    (loss / ACCUM_STEPS).backward()
    if FGM_EPS > 0:
        backup = fgm_attack(model)
        attacked = compute_loss(forward_logits(model, rows), target_tensor, run["weight"])
        (attacked / ACCUM_STEPS).backward()
        model.get_input_embeddings().weight.data.copy_(backup)
    return loss.item()


def optimizer_step(run):
    clip_grad_norm_(run["model"].parameters(), 1.0)
    run["optimizer"].step()
    run["scheduler"].step()
    run["optimizer"].zero_grad()
    run["step"] += 1
    if run["shadow"] is not None:
        ema_update(run["shadow"], run["model"])


print("langkah latih siap | ACCUM_STEPS", ACCUM_STEPS)

In [ ]:
def check_val(run, val_rows, val_y):
    model = run["model"]
    if run["shadow"] is not None:
        ema_swap(run["shadow"], model)
    value = score(val_y, predict_proba(model, val_rows).argmax(axis=1))
    if value > run["best"] or not SELECT_BEST:
        run["best"], run["waited"], run["state"] = value, 0, snapshot(model)
    else:
        run["waited"] += 1
    if run["shadow"] is not None:
        ema_swap(run["shadow"], model)
    run["last_eval"] = run["step"]
    model.train()
    print(f"    langkah {run['step']}: {METRIC_NAME} validasi {value:.4f}")


print("check_val siap | PATIENCE", PATIENCE)

In [ ]:
def build_run(fit_y, seed):
    model = make_model(seed)
    optimizer = torch.optim.AdamW(param_groups(model), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    per_epoch = max(1, int(np.ceil(len(fit_y) / (BATCH_SIZE * ACCUM_STEPS))))
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * per_epoch * EPOCHS),
                                                per_epoch * EPOCHS)
    weight = None
    if CLASS_WEIGHT:
        counts = np.maximum(np.bincount(fit_y, minlength=len(classes)), 1)
        weight = torch.tensor(len(fit_y) / (len(classes) * counts), dtype=torch.float32).to(DEVICE)
    shadow = None
    if EMA_DECAY > 0:
        shadow = {}
        for name, param in model.named_parameters():
            shadow[name] = param.detach().clone()
    return {"model": model, "optimizer": optimizer, "scheduler": scheduler, "weight": weight,
            "shadow": shadow, "step": 0, "best": -1.0, "state": None, "waited": 0,
            "last_eval": 0, "eval_every": max(1, per_epoch // EVALS_PER_EPOCH),
            "rng": np.random.default_rng(seed)}


print("build_run siap")

In [ ]:
def train_epoch(run, fit_rows, fit_y, val_rows, val_y):
    run["model"].train()
    order = run["rng"].permutation(len(fit_rows))
    for number, start in enumerate(range(0, len(order), BATCH_SIZE)):
        chosen = order[start:start + BATCH_SIZE]
        micro_step(run, pick(fit_rows, chosen), fit_y[chosen])
        if (number + 1) % ACCUM_STEPS == 0:
            optimizer_step(run)
            if run["step"] % run["eval_every"] == 0:
                check_val(run, val_rows, val_y)
            if PATIENCE > 0 and run["waited"] >= PATIENCE:
                return True
    return False


print("train_epoch siap")

In [ ]:
def train_fold(fit_rows, fit_y, val_rows, val_y, seed):
    run = build_run(fit_y, seed)
    epoch_seconds = []
    for epoch in range(EPOCHS):
        start = time.time()
        stop = train_epoch(run, fit_rows, fit_y, val_rows, val_y)
        if run["last_eval"] != run["step"]:
            check_val(run, val_rows, val_y)
        epoch_seconds.append(round(time.time() - start, 1))
        print(f"  seed {seed} epoch {epoch + 1}/{EPOCHS}: {epoch_seconds[-1]} dtk "
              f"(termasuk validasi) | terbaik {run['best']:.4f}")
        if stop:
            print("  berhenti dini (PATIENCE)")
            break
    run["model"].load_state_dict(run["state"])
    val_proba = predict_proba(run["model"], val_rows)
    return val_proba, predict_proba(run["model"], test_rows), epoch_seconds


print("train_fold siap")

In [ ]:
def run_fold(number, oof, test_proba, folds_total):
    fit_index, val_index = folds[number]
    fit_rows = pick(train_rows, fit_index) + extra_rows
    fit_y = np.concatenate([y[fit_index], extra_y])
    val_rows = pick(train_rows, val_index)
    seconds = []
    for seed in SEEDS:
        val_p, test_p, epoch_seconds = train_fold(fit_rows, fit_y, val_rows, y[val_index], seed)
        oof[val_index] += val_p / len(SEEDS)
        test_proba += test_p / (len(SEEDS) * folds_total)
        seconds = seconds + epoch_seconds
        torch.cuda.empty_cache()
    return score(y[val_index], oof[val_index].argmax(axis=1)), seconds


print("run_fold siap | seed:", SEEDS)

## Latih k-fold
Tiap fold: latih di bagian latih, prediksi validasi (OOF) dan seluruh test. Test = rata-rata model
tiap fold (tidak dilatih ulang di semua data). Dicetak: detik/epoch, perkiraan sisa, VRAM puncak.

In [ ]:
folds_to_run = FOLDS_TO_RUN
if folds_to_run is None:
    folds_to_run = list(range(len(folds)))
oof = np.zeros((len(train), len(classes)), dtype="float32")
test_proba = np.zeros((len(test), len(classes)), dtype="float32")
fold_scores = []
all_seconds = []
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
began = time.time()
for position, number in enumerate(folds_to_run):
    fold_score, seconds = run_fold(number, oof, test_proba, len(folds_to_run))
    fold_scores.append(fold_score)
    all_seconds = all_seconds + seconds
    left = (time.time() - began) / (position + 1) * (len(folds_to_run) - position - 1)
    print(f"fold {number}: {METRIC_NAME} {fold_score:.4f} | sisa ± {left / 60:.1f} menit")
peak_gb = 0.0
if DEVICE == "cuda":
    peak_gb = torch.cuda.max_memory_allocated() / 2**30
total_seconds = round(time.time() - began, 1)
print(f"rata-rata fold: {np.mean(fold_scores):.4f} +- {np.std(fold_scores):.4f}")
print(f"detik/epoch rata-rata {np.mean(all_seconds):.1f} | VRAM puncak {peak_gb:.2f} GB | "
      f"total {total_seconds / 60:.1f} menit")

## Evaluasi OOF
Skor dihitung di baris train yang punya prediksi OOF (semua baris bila semua fold jalan).

In [ ]:
filled = oof.sum(axis=1) > 0
oof_pred = oof.argmax(axis=1)
oof_score = score(y[filled], oof_pred[filled])
print(f"{METRIC_NAME} OOF ({filled.sum()} baris): {oof_score:.4f}")
print(classification_report(y[filled], oof_pred[filled], target_names=classes.astype(str)))
ConfusionMatrixDisplay.from_predictions(y[filled], oof_pred[filled], display_labels=classes,
                                        normalize="true", xticks_rotation=45)
plt.show()

## Simpan OOF dan catat percobaan
OOF disimpan hanya bila lengkap (semua baris train terisi). `percobaan.csv` bertambah satu baris
tiap run, juga untuk run sebagian fold.

In [ ]:
if filled.all():
    np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_proba)
    print("OOF dan test disimpan:", MODEL_NAME)
else:
    print("OOF tidak lengkap (FOLDS_TO_RUN?): tidak disimpan")
if split_name != "StratifiedKFold":
    print("PERHATIAN: fold bukan StratifiedKFold standar, OOF ini tidak sefold dengan model lain")
note = (f"{os.path.basename(PRETRAINED)} | {split_name} | fold {folds_to_run} | epoch {EPOCHS} "
        f"batch {BATCH_SIZE}x{ACCUM_STEPS} lr {LEARNING_RATE} len {MAX_LENGTH} | "
        f"detik/epoch {np.mean(all_seconds):.1f} | VRAM {peak_gb:.2f} GB | seeds {SEEDS}")
row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": "finetune_oof", "model": MODEL_NAME,
       "metrik": METRIC_NAME, "skor_cv": round(oof_score, 4), "detik": total_seconds,
       "catatan": note}
log_path = f"{OOF_FOLDER}/percobaan.csv"
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(5).to_string())

## Submission
Kolom id diambil dari `sample_submission`; prediksi dipetakan lewat id.

In [ ]:
test_pred = test_proba.argmax(axis=1)
pred_by_id = pd.Series(classes[test_pred], index=test[ID_COL])

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUBMIT_COL] = sample[ID_COL].map(pred_by_id)
# soal minta probabilitas per kelas: ganti baris di atas dengan loop
# proba_by_id = pd.DataFrame(test_proba, columns=classes.astype(str), index=test[ID_COL])
# proba_by_id = proba_by_id.reindex(sample[ID_COL]).reset_index(drop=True)
# submission = pd.concat([submission[[ID_COL]], proba_by_id], axis=1)
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
predicted_share = submission[SUBMIT_COL].value_counts(normalize=True)
print("sebaran prediksi test:", predicted_share.round(3).to_dict())
submission.head()

## Cek submission terhadap sample_submission
Salinan validator T3 (`arsenal/umum/simpan_muat` #15 dan #16): hanya membaca dan mencetak. Hasil
akhir harus `MASALAH: 0`. Dicek: kolom dan urutannya, jumlah baris, id hilang / asing / kembar /
urutan beda, nilai kosong, tipe kolom ("3" vs "3.0"), label di luar `ALLOWED`.

In [ ]:
SUBMISSION_CSV = f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}"
ID_COLUMN = ID_COL

sample = pd.read_csv(SAMPLE_CSV)
written = pd.read_csv(SUBMISSION_CSV)
problems = []


def report(failed, text):
    if failed:
        problems.append(text)


wanted = sample[ID_COLUMN]
ids = written.get(ID_COLUMN, pd.Series(dtype=wanted.dtype))
lost = set(wanted) - set(ids)
extra = set(ids) - set(wanted)
kept = ids[ids.isin(wanted)].drop_duplicates().tolist()
expected = wanted[wanted.isin(ids)].tolist()
report(written.columns.tolist() != sample.columns.tolist(), "kolom / urutan kolom beda")
report(len(written) != len(sample), f"jumlah baris {len(written)}, sample {len(sample)}")
report(lost, f"id hilang {len(lost)}, mis. {sorted(lost)[:3]}")
report(extra, f"id asing {len(extra)}, mis. {sorted(extra)[:3]}")
report(ids.duplicated().any(), f"id kembar: {ids.duplicated().sum()} baris")
report(kept != expected, "urutan id beda dari sample")

In [ ]:
LABEL_COLUMNS = [SUBMIT_COL]     # kolom isi kelas (soal proba per kelas: isi [] untuk lewati cek)
ALLOWED = list(classes)          # label sah = label train, [] = lewati cek

for column in sample.columns:
    if column in written.columns:
        empty = written[column].isna().sum()
        report(empty, f"{column}: {empty} nilai kosong")
        report(written[column].dtype != sample[column].dtype,
               f"{column}: tipe {written[column].dtype}, sample {sample[column].dtype}")
for column in LABEL_COLUMNS:
    if ALLOWED and column in written.columns:
        foreign = set(written[column].dropna()) - set(ALLOWED)
        report(foreign, f"{column}: label di luar ALLOWED {sorted(foreign)[:5]}")
        print(written[column].value_counts())   # semua satu kelas -> curigai model
for problem in problems:
    print("-", problem)
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    print("PERINGATAN: SAMPLE_CSV masih file demo, format panitia belum dicek")
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)
Hanya untuk data latihan yang punya kunci (`ANSWER_CSV` di Konfigurasi). Hapus sel ini dan
baris `ANSWER_CSV` saat lomba.

In [ ]:
answers = pd.read_csv(ANSWER_CSV)
truth_by_id = pd.Series(answers[TARGET].to_numpy(), index=answers[ID_COL])
truth_index = np.searchsorted(classes, truth_by_id.reindex(submission[ID_COL]).to_numpy())
pred_index = np.searchsorted(classes, submission[SUBMIT_COL].to_numpy())
print(f"{METRIC_NAME} di test (kunci): {score(truth_index, pred_index):.4f}")
print(f"{METRIC_NAME} OOF train: {oof_score:.4f}")